# Translation Pipeline — `iknl`

**Source**: `iknl_master_table.csv`  
**Output**: `iknl_master_table_translated.csv`  
**Trustworthiness** (from Makhotso's matrix): `3.8`  
**Volume**: 23 rows · ~390k Dutch chars · ~18 min

### How to run

1. Upload the source CSV to the same folder as this notebook (or change the path below)
2. Run all cells top to bottom
3. The output CSV will appear next to the source, with `_en` columns added

Each translated column is added as a sibling next to the Dutch original. Empty cells, URLs, hashes and dates are skipped automatically.

## 1. Install dependencies

In [1]:
pip install --quiet deep-translator pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Shared translation library

*(Inlined so the notebook is self-contained — same code as `translate_lib.py`.)*

In [2]:
"""
Shared translation library for CSN Team 23 — Dutch → English.

Design choices (from facilitator session + earlier IKNL pipeline):
- deep-translator (Google backend), free tier
- Sentence-aware chunking at 4500 chars (under Google's 5000-char cap)
- 0.5s polite delay between API calls
- Skip cells that are empty, URLs, hashes, dates, or pure numbers
- Cache translations on content_hash to avoid re-translating identical Dutch
- Add a `_en` column next to each translated Dutch column, preserving source structure
"""
import re
import time
import hashlib
from typing import List, Optional
import pandas as pd
from deep_translator import GoogleTranslator

MAX_CHUNK = 4500          # under Google's 5000-char limit
DELAY_SEC = 0.5           # polite delay
RETRY = 3                 # retries per chunk on transient failure

# In-memory cache: maps SHA-1 of Dutch text → English translation
_cache: dict = {}


def _is_translatable(text) -> bool:
    """Return True if the cell holds Dutch text worth translating."""
    if pd.isna(text):
        return False
    s = str(text).strip()
    if not s:
        return False
    # Skip URLs, hashes, ISO dates, pure numbers
    if s.startswith(("http://", "https://", "www.")):
        return False
    if re.fullmatch(r"[0-9a-f]{32,128}", s):  # content hash
        return False
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", s):  # ISO date
        return False
    if re.fullmatch(r"[\d\s.,+\-%]+", s):  # numbers
        return False
    if len(s) < 3:
        return False
    return True


def _split_into_sentences(text: str) -> List[str]:
    """Split on sentence boundaries while preserving punctuation."""
    # Split on . ! ? followed by whitespace
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]


def _chunk_text(text: str, max_chars: int = MAX_CHUNK) -> List[str]:
    """Group sentences into chunks ≤ max_chars."""
    if len(text) <= max_chars:
        return [text]
    sentences = _split_into_sentences(text)
    chunks: List[str] = []
    buf = ""
    for s in sentences:
        if len(buf) + len(s) + 1 <= max_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append(buf)
            # Hard-split sentences that exceed the cap on their own
            if len(s) > max_chars:
                for i in range(0, len(s), max_chars):
                    chunks.append(s[i:i + max_chars])
                buf = ""
            else:
                buf = s
    if buf:
        chunks.append(buf)
    return chunks


def _translate_chunk(chunk: str) -> str:
    """Translate a single chunk with retries."""
    last_err = None
    for attempt in range(RETRY):
        try:
            return GoogleTranslator(source="nl", target="en").translate(chunk)
        except Exception as e:
            last_err = e
            time.sleep(1.0 * (attempt + 1))
    # All retries failed — return original with marker so we know
    return f"[TRANSLATION_FAILED: {last_err}] {chunk}"


def translate_text(text) -> Optional[str]:
    """Public API: translate one cell (string or NaN) → English or None."""
    if not _is_translatable(text):
        return text  # pass through unchanged
    s = str(text).strip()
    # Cache lookup
    key = hashlib.sha1(s.encode("utf-8")).hexdigest()
    if key in _cache:
        return _cache[key]
    # Chunk → translate → reassemble
    chunks = _chunk_text(s)
    translated_chunks = []
    for c in chunks:
        translated_chunks.append(_translate_chunk(c))
        time.sleep(DELAY_SEC)
    out = " ".join(translated_chunks)
    _cache[key] = out
    return out


def translate_dataframe(df: pd.DataFrame,
                        columns_to_translate: List[str],
                        progress_every: int = 10) -> pd.DataFrame:
    """
    Translate specified columns in df, adding `<col>_en` siblings.
    Preserves original Dutch columns untouched.
    """
    out = df.copy()
    total_cells = sum(out[c].notna().sum() for c in columns_to_translate)
    done = 0
    t0 = time.time()
    for col in columns_to_translate:
        en_col = f"{col}_en"
        translations = []
        for idx, val in out[col].items():
            translations.append(translate_text(val))
            done += 1
            if done % progress_every == 0:
                elapsed = time.time() - t0
                rate = done / max(elapsed, 1e-6)
                eta = (total_cells - done) / max(rate, 1e-6)
                print(f"  [{done}/{total_cells}] cells | "
                      f"elapsed {elapsed:.0f}s | ETA {eta:.0f}s | "
                      f"cache_size={len(_cache)}")
        out[en_col] = translations
    print(f"  ✅ Done {total_cells} cells in {time.time() - t0:.0f}s "
          f"(cache hits saved time on repeats)")
    return out


def get_cache_stats() -> dict:
    return {"cached_strings": len(_cache)}


## 3. Configure paths and columns

Change `SRC` if your file lives elsewhere. `COLUMNS` lists the Dutch-text columns to translate — based on the schema-fit validation we did before starting.

In [4]:
SRC = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/iknl_master_table.csv'
DST = '/Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/iknl_master_table_translated.csv'
COLUMNS = ['General Description', 'Decision-making', 'Treatment', 'Statistics & Survival', 'Life after cancer', 'Palliative phase', 'Research', 'Hyperlinks Preserved']
TRUSTWORTHINESS = 3.8   # from Makhotso's source-selection matrix


## 4. Load the source CSV

In [5]:
import pandas as pd
df = pd.read_csv(SRC)
print(f'Loaded {SRC}: {df.shape[0]} rows × {df.shape[1]} cols')
print('Columns:', list(df.columns))
df.head(2)


Loaded /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /CSV Folder/iknl_master_table.csv: 23 rows × 15 cols
Columns: ['Cancer Type', 'Source URL', 'Source Organisation', 'General Description', 'Decision-making', 'Treatment', 'Statistics & Survival', 'Life after cancer', 'Palliative phase', 'Research', 'Scrape Date', 'Content Hash', 'Body Length Chars', 'Hyperlinks Preserved', 'Notes']


,Cancer Type,Source URL,Source Organisation,General Description,Decision-making,Treatment,Statistics & Survival,Life after cancer,Palliative phase,Research,Scrape Date,Content Hash,Body Length Chars,Hyperlinks Preserved,Notes
0,Baarmoederhalskanker,https://iknl.nl/kankersoorten/baarmoederhalska...,IKNL (Integraal Kankercentrum Nederland),Per jaar krijgen rond de 900 vrouwen in Nederl...,Oncoguide biedt zorgprofessionals inzichten ui...,De behandeling van baarmoederhalskanker ofwel ...,"De pagina die u heeft opgevraagd, is niet gevo...","De pagina die u heeft opgevraagd, is niet gevo...",Met goede zorg en begeleiding kunnen zorgprofe...,Gegevens uit de NKR zijn een belangrijke bron ...,2026-05-30,d6da33398298,15824,Kankersoorten -> https://iknl.nl/Kankersoorten...,All sub-pages scraped successfully
1,Baarmoederkanker,https://iknl.nl/kankersoorten/baarmoederkanker,IKNL (Integraal Kankercentrum Nederland),Baarmoederkanker is de meest voorkomende gynae...,"De pagina die u heeft opgevraagd, is niet gevo...","De pagina die u heeft opgevraagd, is niet gevo...",In de Nederlandse Kankerregistratie (NKR) zijn...,"De pagina die u heeft opgevraagd, is niet gevo...",Met goede zorg en begeleiding kunnen zorgprofe...,"De pagina die u heeft opgevraagd, is niet gevo...",2026-05-30,3e62f7b458c2,8566,Kankersoorten -> https://iknl.nl/Kankersoorten...,All sub-pages scraped successfully


## 5. Translate Dutch → English

The pipeline:
1. **Filter** — skip empties, URLs, hashes, dates
2. **Chunk** — split long text on sentence boundaries (≤ 4500 chars per chunk)
3. **Translate** — call Google Translate via deep-translator
4. **Cache** — identical strings translated only once
5. **Reassemble** — chunks joined back into the cell
6. **Side-by-side** — Dutch column preserved; English added as `<col>_en`

In [6]:
translated = translate_dataframe(df, COLUMNS, progress_every=25)
print()
print(f'Output shape: {translated.shape}')
print(f'New columns added: {[c for c in translated.columns if c.endswith("_en")]}')


  [25/182] cells | elapsed 60s | ETA 374s | cache_size=25
  [50/182] cells | elapsed 74s | ETA 196s | cache_size=32
  [75/182] cells | elapsed 119s | ETA 170s | cache_size=48
  [100/182] cells | elapsed 159s | ETA 130s | cache_size=66
  [125/182] cells | elapsed 196s | ETA 89s | cache_size=84
  [150/182] cells | elapsed 236s | ETA 50s | cache_size=104
  [175/182] cells | elapsed 282s | ETA 11s | cache_size=127
  ✅ Done 182 cells in 299s (cache hits saved time on repeats)

Output shape: (23, 23)
New columns added: ['General Description_en', 'Decision-making_en', 'Treatment_en', 'Statistics & Survival_en', 'Life after cancer_en', 'Palliative phase_en', 'Research_en', 'Hyperlinks Preserved_en']


## 6. Spot-check translation quality

In [7]:
for col in COLUMNS:
    en_col = f'{col}_en'
    if en_col not in translated.columns:
        continue
    sample = translated[[col, en_col]].dropna().head(1)
    if sample.empty:
        continue
    print('=' * 60)
    print(f'COLUMN: {col}')
    print(f'NL: {str(sample[col].iloc[0])[:200]}')
    print(f'EN: {str(sample[en_col].iloc[0])[:200]}')
    print()


COLUMN: General Description
NL: Per jaar krijgen rond de 900 vrouwen in Nederland voor het eerst de diagnose baarmoederhalskanker. Baarmoederhalskanker komt opvallend vaak bij jongere vrouwen voor: de piekincidentie ligt tussen het 
EN: Every year, around 900 women in the Netherlands are diagnosed with cervical cancer for the first time. Cervical cancer occurs remarkably often in younger women: the peak incidence is between the ages 

COLUMN: Decision-making
NL: Oncoguide biedt zorgprofessionals inzichten uit richtlijnen, de Nederlandse Kankerregistratie (NKR) en predictiemodellen. Op basis van patiënt- en ziektekenmerken toont Oncoguide stap voor stap een ro
EN: Oncoguide offers healthcare professionals insights from guidelines, the Dutch Cancer Registry (NKR) and prediction models. Based on patient and disease characteristics, Oncoguide shows a step-by-step 

COLUMN: Treatment
NL: De behandeling van baarmoederhalskanker ofwel cervixcarcinoom is, zoals voor bijna elke vorm van kanker,

## 7. Add trustworthiness column and save

Hard-codes the trustworthiness score (since none of the source files contained it). Saves a UTF-8 CSV with all original columns + new `_en` columns.

In [8]:
translated['trustworthiness'] = TRUSTWORTHINESS
translated.to_csv(DST, index=False, encoding='utf-8')
print(f'💾 Saved {DST}')
print(f'   Rows: {translated.shape[0]}')
print(f'   Cols: {translated.shape[1]}')
print(f'   Cache hits during this run: {get_cache_stats()}')


💾 Saved /Users/latafat/Desktop/LSE Data Analytics Program/Employer Project /Translation /Translated_output/iknl_master_table_translated.csv
   Rows: 23
   Cols: 24
   Cache hits during this run: {'cached_strings': 136}


## ✅ Done

Hand the resulting CSV to Quadry for inclusion in the consolidated `csn_knowledge` table.  
The Dutch columns are preserved alongside the English ones so any reviewer can spot-check translations.